In [444]:
import numpy as np
import pandas as pd
import csv
import requests
import xml.etree.ElementTree as ET
from math import atan, exp, cos
from pyproj import Transformer

In [445]:

def vissim_coords_to_lat_long(xVissim, yVissim):
    # variables from vissim net settings
    xRefNet = 0.000
    yRefNet = 0.000

    #Shallowford
    xRefMap = -9468636.559
    yRefMap = 4164801.456
    # constant values
    PI = 3.14159265358979
    EarthRadius = 6378137
    CorrectionFactorMercator = 1.0011202320000001

    # deriving vissim local scal factor
    LatitudeRefPointMap = ( 2 * atan( exp( CorrectionFactorMercator * yRefMap / EarthRadius ) ) - PI / 2 ) / ( PI / 180 )
    LocalScaleFactor = 1 / cos( LatitudeRefPointMap * PI / 180 )

    # calculating xy coordinates in Mercator
    xMercator = ( xVissim - xRefNet ) * LocalScaleFactor + xRefMap
    yMercator = ( yVissim - yRefNet ) * LocalScaleFactor + yRefMap

    # transform Mercator coordinates to WGS84
    merc2wgs84 = Transformer.from_crs('ESRI:53004', 'EPSG:4326')
    wgs84 = merc2wgs84.transform(xMercator, yMercator)
    # print(wgs84)
    p = [wgs84[0], wgs84[1]]
    # print (p)
    return (p)

In [379]:
xml_file_name ="C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\shallowford_mod_v9_signals_PM.inpx"

In [386]:
from xml.dom import minidom
import xml.dom.minidom
dom = minidom.parse(xml_file_name)

tree = ET.parse(xml_file_name)
# print (tree)

root = tree.getroot()
# print (root)

big_list_link_info = []
for child in root.findall(".//links//link"):
#     print (child.attrib)
    list_link_info = [child.attrib['no'],child.attrib['name'], child.attrib['isPedArea']]
    connector_info = []
#     print ("HERE")
    if int(child.attrib['no'])>=10000:
        list_link_info.append("possibly_connector")
    else: 
        list_link_info.append("possibly_link")
        
    for another_child in child:
#         print (another_child.tag)
        if another_child.tag == 'fromLinkEndPt':
            connector_info.append(another_child.attrib['lane'])
            connector_info.append(another_child.attrib['pos'])
            
        if another_child.tag == 'toLinkEndPt':
            connector_info.append(another_child.attrib['lane'])
            connector_info.append(another_child.attrib['pos'])

    if len(connector_info)<1:
        na_info = ['NA','NA','NA','NA']
        connector_info=na_info
    
    list_link_info = list_link_info + connector_info    

    # print (list_link_info)
    big_list_link_info.append(list_link_info)

big_list_link_coord_info = []
for grand_child in root.findall(".//links//geometry//linkPolyPts//linkPolyPoint[1]"):
    list_link_coord_info = [grand_child.attrib["x"],grand_child.attrib["y"],grand_child.attrib["zOffset"]]
    big_list_link_coord_info.append(list_link_coord_info)



In [387]:
fin_big_list = []
for i in range(len(big_list_link_coord_info)):
    k = big_list_link_info[i]+big_list_link_coord_info[i]
    # print (k)
    fin_big_list.append(k)

In [388]:
df = pd.DataFrame(fin_big_list, columns = ['Link_No', 'Link_Name', 'Ped_Area','Link-or-Connector','FromLinkLane','FromLinkPos','ToLinkLane','ToLinkPos','StartLinkCoord-X', 'StartLinkCoord-Y', 'StartLinkCoord-Z'])
# print (df)
df.to_csv('C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\vissim_link_detailed_info_2.csv', index=False)
file_name3 = 'C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\vissim_link_detailed_info_2.csv'
df_3 = pd.read_csv(file_name3)
# list_junction_ids_only = df_2['junction_id'].unique().tolist()
df_3['Latitude'] = vissim_coords_to_lat_long(df_3['StartLinkCoord-X'],df_3['StartLinkCoord-Y'])[0]
df_3['Longitude'] = vissim_coords_to_lat_long(df_3['StartLinkCoord-X'],df_3['StartLinkCoord-Y'])[1]
# print (df_3)
df_3.to_csv('C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\vissim_link_detailed_info_2_lat_long.csv',index=False)

In [389]:
df3 = pd.read_csv('C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\vissim_link_detailed_info_2_lat_long.csv')
connector_df = df3[df3['Link-or-Connector'] == 'possibly_connector'] 

In [390]:
def if_conn_from_div_link(row):
    # filter for connector only
    fromlink = row['FromLinkLane'].split(" ")[0]
    fromlane = row['FromLinkLane'].split(" ")[1]
    tolink = row['ToLinkLane'].split(" ")[0]
    tolane = row['ToLinkLane'].split(" ")[1]
    p = [fromlink, fromlane, tolink, tolane]
    return p

connector_df['FromLink'] = connector_df.apply(lambda row: if_conn_from_div_link(row)[0], axis=1)
connector_df['FromLane'] = connector_df.apply(lambda row: if_conn_from_div_link(row)[1], axis=1)
connector_df['ToLink'] = connector_df.apply(lambda row: if_conn_from_div_link(row)[2], axis=1)
connector_df['ToLane'] = connector_df.apply(lambda row: if_conn_from_div_link(row)[3], axis=1)
# print (connector_df.head())

C:\Users\ets\AppData\Local\Temp\2\ipykernel_21540\51284203.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  connector_df['FromLink'] = connector_df.apply(lambda row: if_conn_from_div_link(row)[0], axis=1)
C:\Users\ets\AppData\Local\Temp\2\ipykernel_21540\51284203.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  connector_df['FromLane'] = connector_df.apply(lambda row: if_conn_from_div_link(row)[1], axis=1)
C:\Users\ets\AppData\Local\Temp\2\ipykernel_21540\51284203.py:12: SettingWithCopyWarning: 
A v

In [391]:
# function to find if connector is coming from a divergence link 
def is_link_div(row):
    from_link_no  = row['FromLink']
    #filter for this from link
    filter_from_link_connector_df = connector_df[connector_df['FromLink'] == from_link_no]
    if filter_from_link_connector_df.shape[0]>1:
        if len(filter_from_link_connector_df.ToLink.unique().tolist())==filter_from_link_connector_df.shape[0]:        
            p = ["divergence", "divergence-"+from_link_no]
        else:
            print ("duplication possible for - ", from_link_no)
            p = ["no divergence but merge case", "possible merge" ]
    else:
        p = ["no divergence", "no divergence"]
    return p

In [392]:
connector_df['Divergence'] = connector_df.apply(lambda row: is_link_div(row)[0], axis=1)
connector_df['Divergence_Name'] = connector_df.apply(lambda row: is_link_div(row)[1], axis=1)
# print (connector_df.head())

C:\Users\ets\AppData\Local\Temp\2\ipykernel_21540\747710920.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  connector_df['Divergence'] = connector_df.apply(lambda row: is_link_div(row)[0], axis=1)
C:\Users\ets\AppData\Local\Temp\2\ipykernel_21540\747710920.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  connector_df['Divergence_Name'] = connector_df.apply(lambda row: is_link_div(row)[1], axis=1)


In [393]:
connector_df_div_only = connector_df[connector_df['Divergence']== 'divergence']
# print (connector_df_div_only)
div_link_list = connector_df_div_only.FromLink.unique().tolist()

# print (div_link_list)

In [394]:
links_df = df[df['Link-or-Connector']== 'possibly_link']
# print (links_df)
links_df["Divergence"] = np.where(links_df["Link_No"].isin(div_link_list), "divergence", "no divergence")
# print (links_df.head())
links_df.to_csv('C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\links_divergence_inventory.csv', index=False)
links_df2 = pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\links_divergence_inventory.csv")
links_df2['Latitude'] = vissim_coords_to_lat_long(links_df2['StartLinkCoord-X'],links_df2['StartLinkCoord-Y'])[0]
links_df2['Longitude'] = vissim_coords_to_lat_long(links_df2['StartLinkCoord-X'],links_df2['StartLinkCoord-Y'])[1]
links_df2["FromLink"] = " "
links_df2["FromLane"] = " "
links_df2["ToLink"] = " "
links_df2["ToLane"] = " "
links_df2["Divergence_Name"] =" "
links_df2 = links_df2[['Link_No', 'Link_Name', 'Ped_Area', 'Link-or-Connector', 'FromLinkLane', 'FromLinkPos', 'ToLinkLane', 'ToLinkPos', 'StartLinkCoord-X', 'StartLinkCoord-Y', 'StartLinkCoord-Z', 'Latitude',  'Longitude', 'FromLink', 'FromLane', 'ToLink', 'ToLane',   'Divergence', 'Divergence_Name']]

# print (links_df2.head())
# print (connector_df.head())
# links_df.to_csv('C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\links_divergence_inventory.csv', index=False)
df_combined = pd.concat([links_df2, connector_df], axis=0)
print (df_combined.head())
df_combined.to_csv('C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\combined_inventory.csv', index=False)

   Link_No    Link_Name  Ped_Area Link-or-Connector FromLinkLane  FromLinkPos  \
0        1  280-0-Right     False     possibly_link          NaN          NaN   
1        2  281-0-Right     False     possibly_link          NaN          NaN   
2        3  282-0-Right     False     possibly_link          NaN          NaN   
3        4  283-0-Right     False     possibly_link          NaN          NaN   
4        5  284-0-Right     False     possibly_link          NaN          NaN   

  ToLinkLane  ToLinkPos  StartLinkCoord-X  StartLinkCoord-Y  StartLinkCoord-Z  \
0        NaN        NaN        683.928073       -242.442229                 0   
1        NaN        NaN        171.685256        -15.358512                 0   
2        NaN        NaN        170.006919        -32.948713                 0   
3        NaN        NaN        184.079533         -9.434332                 0   
4        NaN        NaN        181.725205        -28.568434                 0   

    Latitude  Longitude Fr

C:\Users\ets\AppData\Local\Temp\2\ipykernel_21540\1822441443.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  links_df["Divergence"] = np.where(links_df["Link_No"].isin(div_link_list), "divergence", "no divergence")


In [395]:
connector_df = df_combined[df_combined['Link-or-Connector']=='possibly-connector']

In [396]:
def find_next_and_previous_link_or_connector(roadid):
    #filter for the road id

    if roadid<10000:
        df_temp = df_combined[df_combined['FromLink']==str(roadid)]
        # print (df_temp)
        next_road_list = df_temp['Link_No'].tolist()
        df_temp2 = df_combined[df_combined['ToLink']==str(roadid)]
        prev_road_list = df_temp2['Link_No'].tolist()
        # from_link = df_combined.loc[df_combined['Link_No'] == roadid, 'FromLink'].iloc[0]
        # print (next_road_list)
        # print (prev_road_list)

    if roadid>=10000:
        df_temp = df_combined[df_combined['Link_No']==roadid]
        next_road_list = df_temp['ToLink'].tolist()
        next_road_list = [eval(i) for i in next_road_list]
        prev_road_list = df_temp['FromLink'].tolist()
        prev_road_list = [eval(i) for i in prev_road_list]
        # print (next_road_list)
        # print (prev_road_list)
    return [next_road_list, prev_road_list]

find_next_and_previous_link_or_connector(30)

[[10061, 10062, 10063, 10064], []]

In [397]:
## This function should be exported to mapping file
def find_next_connector(conn_no):
    #get to link
    df_temp = df_combined[df_combined['Link_No']== conn_no]
    next_road_list = df_temp['ToLink'].tolist()
    to_link = next_road_list[0]
    print (to_link)
    # next_road_list = [eval(i) for i in next_road_list]

    # to_link = connector_df.loc[connector_df['Link_No'] == conn_no, 'ToLink'].iloc[0]
    
    #filter for this from link
    filter_from_link_connector_df = df_combined[df_combined['FromLink'] == to_link]
    
    conn_list = filter_from_link_connector_df['Link_No'].to_list()
    print(conn_list)        
    return (conn_list)

    
    # get list of connectors associated with this
find_next_connector(10061) 

98
[10180]


[10180]

In [398]:
## This function should be exported to mapping file
def get_list_of_link_and_connector_till_next_div(st_link_no):
    big_list_of_link_conn_till_div = []
    
    
    # filter_from_link_connector_df = df_combined[df_combined['FromLink'] == st_link_no]
    filter_from_link_connector_df = df_combined[df_combined['FromLink'] == str(st_link_no)]
    print (filter_from_link_connector_df)
    list_of_conns = filter_from_link_connector_df['Link_No'].to_list() 
    print (list_of_conns)
    
    for j in range(len(list_of_conns)):
        i = 0
        
        list_of_link_conn_till_div = [st_link_no]
        
        conn_no = list_of_conns[j]
        list_of_link_conn_till_div.append(conn_no)
        
        while i == 0:
            df_temp = df_combined[df_combined['Link_No']== conn_no]
            next_road_list = df_temp['ToLink'].tolist()
            to_link = next_road_list[0]
            # to_link = connector_df.loc[connector_df['Link_No'] == conn_no, 'ToLink'].iloc[0]
             
            list_of_link_conn_till_div.append(to_link)

            if len(find_next_connector(conn_no)) ==1:
                list_of_link_conn_till_div.append(find_next_connector(conn_no)[0])
                conn_no = find_next_connector(conn_no)[0]

            elif len(find_next_connector(conn_no)) ==0:
                list_of_link_conn_till_div.append("no connector ahead")
                i = 1
            else:    
                list_of_link_conn_till_div.append("reached next divergence point")
                i = 1
        
        big_list_of_link_conn_till_div.append(list_of_link_conn_till_div)
        
    return (big_list_of_link_conn_till_div)
            
get_list_of_link_and_connector_till_next_div(30)

     Link_No Link_Name  Ped_Area   Link-or-Connector FromLinkLane  \
224    10061       NaN     False  possibly_connector         30 1   
225    10062       NaN     False  possibly_connector         30 1   
226    10063       NaN     False  possibly_connector         30 1   
227    10064       NaN     False  possibly_connector         30 1   

     FromLinkPos ToLinkLane  ToLinkPos  StartLinkCoord-X  StartLinkCoord-Y  \
224   385.670623       98 1        0.4       -425.771529        235.820284   
225   385.670623       99 1        0.4       -425.771529        235.820284   
226   385.670623      100 1        0.4       -425.771529        235.820284   
227   385.670623      101 1        0.4       -425.771529        235.820284   

     StartLinkCoord-Z   Latitude  Longitude FromLink FromLane ToLink ToLane  \
224                 0  35.043214 -85.158171       30        1     98      1   
225                 0  35.043214 -85.158171       30        1     99      1   
226                 0  35.

[[30, 10061, '98', 10180, '14', 'reached next divergence point'],
 [30, 10062, '99', 10181, '17', 'no connector ahead'],
 [30, 10063, '100', 10182, '16', 'no connector ahead'],
 [30, 10064, '101', 10183, '15', 'no connector ahead']]

In [399]:
def vissim_link_to_opendrive_id_mapping():

    df_temp = df_combined[df_combined["Link-or-Connector"]=="possibly_link"]
    data = pd.read_csv(r'C:\Users\ets\Desktop\Projects\RT_ScenarioGenerator\Simulation\vissim_dev_tests\shallowford\opendrive_road_info_lat_long_inventory.csv')
    opendrive_road_name_list=[]
    for index, row in df_temp.iterrows():
        road_name = row['Link_Name']
        # print (road_name)
        opendrive_road_name = road_name[:-8]
        if opendrive_road_name[0]==":":
            # print (opendrive_road_name)
            # print (data.loc[data['road_name'] == opendrive_road_name, 'road_id'])
            opendrive_road_id = data.loc[data['road_name'] == opendrive_road_name, 'road_id']
            opendrive_road_name_list.append(opendrive_road_id)
        else:
            opendrive_road_name_list.append(opendrive_road_name)
    df_temp = df_temp.assign(opendrive_road_id=opendrive_road_name_list)
    print (df_temp.head())
    df_temp.to_csv(r"C:\Users\ets\Desktop\Projects\RT_ScenarioGenerator\Simulation\vissim_dev_tests\shallowford\vissim_link_opendrive_road_id_mapping.csv")
    return df_temp

vissim_link_to_opendrive_id_mapping()

# df_temp["corresponding_opendrive_road_id"] = df_temp[""]#modify road name to eliminate last characters



   Link_No    Link_Name  Ped_Area Link-or-Connector FromLinkLane  FromLinkPos  \
0        1  280-0-Right     False     possibly_link          NaN          NaN   
1        2  281-0-Right     False     possibly_link          NaN          NaN   
2        3  282-0-Right     False     possibly_link          NaN          NaN   
3        4  283-0-Right     False     possibly_link          NaN          NaN   
4        5  284-0-Right     False     possibly_link          NaN          NaN   

  ToLinkLane  ToLinkPos  StartLinkCoord-X  StartLinkCoord-Y  StartLinkCoord-Z  \
0        NaN        NaN        683.928073       -242.442229                 0   
1        NaN        NaN        171.685256        -15.358512                 0   
2        NaN        NaN        170.006919        -32.948713                 0   
3        NaN        NaN        184.079533         -9.434332                 0   
4        NaN        NaN        181.725205        -28.568434                 0   

    Latitude  Longitude Fr

,Link_No,Link_Name,Ped_Area,Link-or-Connector,FromLinkLane,FromLinkPos,ToLinkLane,ToLinkPos,StartLinkCoord-X,StartLinkCoord-Y,StartLinkCoord-Z,Latitude,Longitude,FromLink,FromLane,ToLink,ToLane,Divergence,Divergence_Name,opendrive_road_id
0,1,280-0-Right,False,possibly_link,NaN,NaN,NaN,NaN,683.928073,-242.442229,0,35.038913,-85.145982,,,,,divergence,,280
1,2,281-0-Right,False,possibly_link,NaN,NaN,NaN,NaN,171.685256,-15.358512,0,35.040955,-85.151609,,,,,divergence,,281
2,3,282-0-Right,False,possibly_link,NaN,NaN,NaN,NaN,170.006919,-32.948713,0,35.040797,-85.151627,,,,,no divergence,,282
3,4,283-0-Right,False,possibly_link,NaN,NaN,NaN,NaN,184.079533,-9.434332,0,35.041009,-85.151472,,,,,no divergence,,283
4,5,284-0-Right,False,possibly_link,NaN,NaN,NaN,NaN,181.725205,-28.568434,0,35.040837,-85.151498,,,,,divergence,,284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,181,:7564605312_5-0-Right,False,possibly_link,NaN,NaN,NaN,NaN,270.426927,-59.273175,0,35.040561,-85.150524,,,,,no divergence,,"180 460 Name: road_id, dtype: int64"
169,182,:7564605312_7-0-Right,False,possibly_link,NaN,NaN,NaN,NaN,270.499474,-60.875245,0,35.040546,-85.150523,,,,,no divergence,,"181 461 Name: road_id, dtype: int64"
170,183,:7564605312_8-0-Right,False,possibly_link,NaN,NaN,NaN,NaN,253.345570,-62.203812,0,35.040534,-85.150712,,,,,no divergence,,"182 462 Name: road_id, dtype: int64"
171,184,:7564605312_10-0-Right,False,possibly_link,NaN,NaN,NaN,NaN,253.990019,-60.736549,0,35.040547,-85.150704,,,,,no divergence,,"183 463 Name: road_id, dtype: int64"


In [400]:

def get_corresponding_vissim_id(opendrive_roadid):
    data = pd.read_csv(r'C:\Users\ets\Desktop\Projects\RT_ScenarioGenerator\Simulation\vissim_dev_tests\shallowford\vissim_link_opendrive_road_id_mapping.csv')
    vissim_road_id = data.loc[data['opendrive_road_id'] == str(opendrive_roadid), 'Link_No'].item()
    # print (vissim_road_id)
    return vissim_road_id

get_corresponding_vissim_id(280)

1

In [401]:

def get_closest_divergent_id(vissim_road_id):
    data2 = pd.read_csv(r'C:\Users\ets\Desktop\Projects\RT_ScenarioGenerator\Simulation\vissim_dev_tests\shallowford\vissim_link_opendrive_road_id_mapping.csv')

    # prev_link = find_next_and_previous_link_or_connector(vissim_road_id)[1][0]
    prev_link = find_next_and_previous_link_or_connector(vissim_road_id)[1]
    # print (prev_link)

    if len(prev_link)>1:
        prev_link ="multiple links lead to this"
        pass

    elif prev_link ==[]:
        # print ("no prev link")
        # prev_link = "no prev link"
        prev_link = vissim_road_id
        # list_of_prev_link.append(prev_link)
        pass
    # print (prev_link)
    # if prev_link == None:
    #     pass

    elif prev_link[0]<10000:
        # print ("here1")
        # print (prev_link[0])
        prev_link = prev_link[0]
        divergence_status = data2.loc[data2['Link_No'] == prev_link, 'Divergence'].item()
        # print (divergence_status)

        if divergence_status == "divergence":
            prev_link ="Unexpected divergent link before"
            pass
        else:
            # print ("here2")
            prev_link= "Unexpected non divergent link before"
            # prev_link = get_prev_road_list(prev_link)
            # list_of_prev_link.append(prev_link)
            
    else:
        # print ("here3")
        prev_link = prev_link[0]
        prev_link = find_next_and_previous_link_or_connector(prev_link)[1][0]
        divergence_status = data2.loc[data2['Link_No'] == prev_link, 'Divergence'].item()

    return prev_link



def get_start_route(id):
    # flag = 0
    k = 0
    p = get_closest_divergent_id(id)
    print (p)
    if p == 'multiple links lead to this':
        start_route = id
        k = 1
    elif p =='no prev link':
        print ("here")
        # start_route = find_next_and_previous_link_or_connector(id)[1]
        start_route = id
        flag = 1
    elif p == "Unexpected divergent link before":
        start_route = "to be determined"
        k = 1
    elif p == "Unexpected non divergent link before":
        start_route = "to be determined"
        k = 1
    else:
        start_route = p
        # later can also check if not divergence and make it recursive
        # data = pd.read_csv(r'C:\Users\ets\Desktop\Projects\RT_ScenarioGenerator\Simulation\vissim_dev_tests\shallowford\vissim_link_opendrive_road_id_mapping.csv')
        # divergence_status = data.loc[data['Link_No'] == start_route, 'Divergence'].item()
        # if divergence_status == "divergence":
        #     start_route = p
        #     k = 1

        # elif divergence_status != "divergence":

        #     while k!=1 and flag!=0: 
        #         print (flag)
        #         get_start_route(p)
    data = pd.read_csv(r'C:\Users\ets\Desktop\Projects\RT_ScenarioGenerator\Simulation\vissim_dev_tests\shallowford\vissim_link_opendrive_road_id_mapping.csv')
    divergence_status = data.loc[data['Link_No'] == start_route, 'Divergence'].item()
    h = find_next_and_previous_link_or_connector(start_route)[1]
    kk = [start_route, divergence_status,h]

    return kk

print (get_start_route(31))


32
[32, 'no divergence', []]


In [402]:
def get_route_end_vissim_id(vissim_road_id):
    connectors = find_next_and_previous_link_or_connector(vissim_road_id)[0]
    print (connectors)
    # connectors = connectors.sort()
    links = []
    print (len(connectors))
    for i in range(len(connectors)):
        next_link = find_next_and_previous_link_or_connector(connectors[i])[0][0]
        links.append(next_link)
    # print(links)
    links.sort()
    print (links)
    return links

get_route_end_vissim_id(31)

[10065, 10066, 10067, 10068]
4
[73, 74, 75, 76]


[73, 74, 75, 76]

In [448]:
from __future__ import print_function
import os
# COM-Server
import win32com.client as com

In [449]:
Vissim = com.gencache.EnsureDispatch("Vissim.Vissim") #
# Vissim =com.Dispatch("Vissim.Vissim")
Path_of_COM_Basic_Commands_network = 'C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford'

## Load a Vissim Network:
Filename               = os.path.join(Path_of_COM_Basic_Commands_network, 'shallowford_mod_v10_signals_PM.inpx')
flag_read_additionally = False # you can read network(elements) additionally, in this case set "flag_read_additionally" to true
Vissim.LoadNet(Filename, flag_read_additionally)

## Load a Layout:
Filename = os.path.join(Path_of_COM_Basic_Commands_network, 'shallowford_mod_v10_signals_PM.layx')
Vissim.LoadLayout(Filename)

#vissim function set a route at a position for a vissim road id 
# vissim_start_link = Vissim.Net.Links.ItemByKey(32)
# VehRoutDesSta = Vissim.Net.VehicleRoutingDecisionsStatic.AddVehicleRoutingDecisionStatic(1, vissim_start_link, 100)
# vissim_end_link1 = Vissim.Net.Links.ItemByKey(73)
# vissim_end_link2 = Vissim.Net.Links.ItemByKey(74)
# VehRoutSta1 = VehRoutDesSta.VehRoutSta.AddVehicleRouteStatic(1, vissim_end_link1, 10)
# VehRoutSta2 = VehRoutDesSta.VehRoutSta.AddVehicleRouteStatic(2, vissim_end_link2, 10)
# VehRoutSta1.SetAttValue("RelFlow(1)", 0.1)
# VehRoutSta2.SetAttValue("RelFlow(1)", 0.9)


In [411]:
#vissim function set a route at a position for a vissim road id 
def set_route(route_no, vissim_start_link_no, vissim_dest_link_list, route_flow_ratio_list, interval_number):
    vissim_start_link = Vissim.Net.Links.ItemByKey(vissim_start_link_no)
    if interval_number==1:
        VehRoutDesSta = Vissim.Net.VehicleRoutingDecisionsStatic.AddVehicleRoutingDecisionStatic(route_no, vissim_start_link, 5)
        for i in range(len(vissim_dest_link_list)):
            vissim_end_link = Vissim.Net.Links.ItemByKey(vissim_dest_link_list[i])
            VehRoutSta = VehRoutDesSta.VehRoutSta.AddVehicleRouteStatic(i+1, vissim_end_link, 4)
            VehRoutSta.SetAttValue("RelFlow("+str(interval_number)+")", route_flow_ratio_list[i])
    else:
        VehRoutSta = Vissim.Net.VehicleRoutingDecisionsStatic.ItemByKey(route_no)
        for i in range(len(vissim_dest_link_list)):
            Vissim.Net.VehicleRoutingDecisionsStatic.ItemByKey(route_no).VehRoutSta.ItemByKey(i+1).SetAttValue("RelFlow("+str(interval_number)+")", route_flow_ratio_list[i])
            # VehRoutSta.SetAttValue("RelFlow("+str(interval_number)+")", route_flow_ratio_list[i])


       
# set_route(1, 32, [73, 74], [0.1,0.9])


In [406]:
## function to create time interval sets for vehicle input and routings 
## TimeIntervalSet = 1   # 1 = VehicleInput

simtime = 3600
interval_duration = 15*60
no_intervals = int(simtime/interval_duration)-1
print (no_intervals)
startTimeofDayinSec = 28800
start_interval = int(startTimeofDayinSec/interval_duration)+1
end_interval = start_interval+no_intervals

def setTimeIntervalSets(no_intervals):
    for i in range(no_intervals):
        Vissim.Net.TimeIntervalSets.ItemByKey(1).TimeInts.AddTimeInterval(0) # unsigned int Key
        # print (Vissim.Net.TimeIntervalSets.ItemByKey(1).TimeInts(i+1))
        # print ("ALL")
        Vissim.Net.TimeIntervalSets.ItemByKey(2).TimeInts.AddTimeInterval(0) # unsigned int Key

# setTimeIntervalSets(no_intervals)


3


In [407]:
## This is a test cell
startTimeofDayinSec = 28800
start_interval = int(startTimeofDayinSec/interval_duration)+1
end_interval = start_interval+no_intervals

print (start_interval)
print (end_interval)

df_gridsmart_routes = pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_lookuptable_2.csv")
df_gridsmart_demand =  pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_demand.csv")
df_gridsmart_routes_temp = df_gridsmart_routes[df_gridsmart_routes["OpenDriveFromID"]==309]
intersection = df_gridsmart_routes_temp["IntersectionName"].unique()
print (intersection)
print (df_gridsmart_routes_temp)
route_movement_list = df_gridsmart_routes_temp["Turn"].unique().tolist()
print (route_movement_list)
df_demand_temp = df_gridsmart_demand[df_gridsmart_demand["IntersectionName"]==intersection[0]]
for j in range(start_interval, end_interval+1):
    demand = df_demand_temp.loc[j-1, route_movement_list[0]]
    print (demand)
    


33
36
['Amin Dr./Shallowford Village Dr. & Shallowford Rd.']
                                    IntersectionName Turn  OpenDriveFromID  \
0  Amin Dr./Shallowford Village Dr. & Shallowford...  NBR            309.0   
1  Amin Dr./Shallowford Village Dr. & Shallowford...  NBT            309.0   
2  Amin Dr./Shallowford Village Dr. & Shallowford...  NBL            309.0   
3  Amin Dr./Shallowford Village Dr. & Shallowford...  NBU            309.0   

   OpenDriveToID  
0          293.0  
1          296.0  
2          295.0  
3          294.0  
['NBR', 'NBT', 'NBL', 'NBU']
5.0
6.0
2.0
6.0


In [408]:
def set_vissim_static_routing(opendrive_id, route_no):
    #get opendrive from id - find the relevant link - check if divergent link - if yes then - get previous road list until a road that is divergent or None-> if the last link is divergent then  get link after one connector -> if not divergent then use the last link as the link for start point
    df_gridsmart_routes = pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_lookuptable_2.csv")
    df_gridsmart_demand =  pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_demand.csv")
    df_gridsmart_routes_temp = df_gridsmart_routes[df_gridsmart_routes["OpenDriveFromID"]==opendrive_id]
    intersection = df_gridsmart_routes_temp["IntersectionName"].unique()
    print (intersection)
    print (df_gridsmart_routes_temp)
    route_movement_list = df_gridsmart_routes_temp["Turn"].unique().tolist()
    print (route_movement_list)

    # for each movement get demand value, get turn as key
    # demand_dict = {}
    
    for j in range(start_interval, end_interval+1):
        demand_list = []
        print (j)
        for i in range(len(route_movement_list)):
            df_demand_temp = df_gridsmart_demand[df_gridsmart_demand["IntersectionName"]==intersection[0]].reset_index()
            # print (df_demand_temp)
            demand = df_demand_temp.loc[j-1, route_movement_list[i]]
           
            print (demand)
            # dict={route_movement_list[i]:demand}
            # demand_dict[route_movement_list[i]].append(demand)
            demand_list.append(demand)
    
        vissim_id = get_corresponding_vissim_id(opendrive_id)
        vissim_start_link_no = get_start_route(vissim_id)[0]
        print (vissim_start_link_no)
        vissim_dest_link_list = get_route_end_vissim_id(vissim_id)
        print ("HERE")
        # print (route_flow_ratio_list)
        route_flow_ratio_list = [0 if str(x)=='nan' else x for x in demand_list]
        print (route_flow_ratio_list)
        sumlist = sum(route_flow_ratio_list)
        route_flow_ratio_list = [x/sumlist for x in route_flow_ratio_list]
        print (route_flow_ratio_list)
        print (vissim_start_link_no)
        print (vissim_dest_link_list)
        print (route_flow_ratio_list)

        set_route(route_no, vissim_start_link_no, vissim_dest_link_list, route_flow_ratio_list, j-start_interval+1)
    

# set_vissim_static_routing(297, 5)



#     # print (df_gridsmart_routes)
#     unique_opendrivefromid_list = df_gridsmart_routes['OpenDriveFromID'].unique()
# # for i in range(len(unique_opendrivefromid_list)):
# for i in range(1):
#     opendrivefromid = unique_opendrivefromid_list[i]
#     opendrivefromid= int(opendrivefromid)

#     #find corresponding vissim link
#     p = get_corresponding_vissim_id(opendrivefromid)

    # find start of route link id


    # route end points [get list of connectors from this divergent link, each connector length - some distance is end position]
    
    
    #


In [409]:
## Set Route for All Divergent Links
def set_route_for_all_div_links():
    df_gridsmart_routes = pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_lookuptable.csv")
    df_gridsmart_demand =  pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_demand.csv")
    print (df_gridsmart_routes)
    #get list of opendrive ids
    unique_opendrive_id_list  = df_gridsmart_routes["OpenDriveFromID"].unique().tolist()
    print (unique_opendrive_id_list)
    unique_opendrive_id_list2 = [x for x in unique_opendrive_id_list if pd.notnull(x)]
    # unique_opendrive_id_list2 = [x for x in unique_opendrive_id_list if x != 'nan']
    print (unique_opendrive_id_list2)

    for i in range(len(unique_opendrive_id_list2)):
        print ("here")
        print (unique_opendrive_id_list2[i])
        set_vissim_static_routing(int(unique_opendrive_id_list2[i]), i+1)
        
    
    



In [412]:
set_route_for_all_div_links()

                                     IntersectionName Turn  OpenDriveFromID  \
0   Amin Dr./Shallowford Village Dr. & Shallowford...  NBR            309.0   
1   Amin Dr./Shallowford Village Dr. & Shallowford...  NBT            309.0   
2   Amin Dr./Shallowford Village Dr. & Shallowford...  NBL            309.0   
3   Amin Dr./Shallowford Village Dr. & Shallowford...  NBU            309.0   
4   Amin Dr./Shallowford Village Dr. & Shallowford...  EBR            312.0   
..                                                ...  ...              ...   
91                   Gunbarrel Road & Shallowford Rd.  SBU            326.0   
92                   Gunbarrel Road & Shallowford Rd.  WBR            280.0   
93                   Gunbarrel Road & Shallowford Rd.  WBT            280.0   
94                   Gunbarrel Road & Shallowford Rd.  WBL            280.0   
95                   Gunbarrel Road & Shallowford Rd.  WBU            280.0   

    OpenDriveToID  
0           293.0  
1          

In [446]:
def list_of_inc_roads_for_a_junction(junction_id):
    file_name = 'C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\opendrive_junction_info.csv'
    df = pd.read_csv (file_name)
    rslt_df = df[df['junction_id'] == junction_id] 
#     print (rslt_df)
    list_inc_roads_for_jid = rslt_df['incoming_road'].unique().tolist()
    print (list_inc_roads_for_jid)
    return list_inc_roads_for_jid

In [447]:
def get_corresponding_opendrive_junctionID(intid):
    synchro_lookup_raw = pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\synchro_lookuptable.csv")
    opendriveJunctionID = synchro_lookup_raw.loc[synchro_lookup_raw['INTID'] == intid, 'OpenDriveJunctionID'].iloc[0]
    # print(opendriveJunctionID)
    return (opendriveJunctionID)

In [415]:
def get_corresponding_incoming_approach_link_numbers(intid):
    jid = get_corresponding_opendrive_junctionID(intid)
    inc_rd_list = list_of_inc_roads_for_a_junction(jid)
    for i in range(len(inc_rd_list)):
        vissim_road_id = get_corresponding_vissim_id(inc_rd_list[i])
        # print(vissim_road_id)
        


In [416]:
get_corresponding_incoming_approach_link_numbers(4)

[324, 298, 309, 312]


In [417]:
def get_corresponding_turn_movement_link_numbers(intid):
    jid = get_corresponding_opendrive_junctionID(intid)
    inc_rd_list = list_of_inc_roads_for_a_junction(jid)
    df_gridsmart_routes = pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_lookuptable_2.csv")
    df_gridsmart_demand =  pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_demand.csv")
    # df_gridsmart_routes_temp = df_gridsmart_routes[df_gridsmart_routes["OpenDriveFromID"]==inc_rd_list[0]]

    # # intersection = df_gridsmart_routes_temp["IntersectionName"].unique()
    # # print (intersection)
    # print (df_gridsmart_routes_temp)
    # route_movement_list = df_gridsmart_routes_temp["Turn"].unique().tolist()
    # print (route_movement_list)

    dict_movement_list=[]
    vissim_approach_links_dict = {}
    for i in range(len(inc_rd_list)):
    # for i in range(1,2):
        df_gridsmart_routes_temp = df_gridsmart_routes[df_gridsmart_routes["OpenDriveFromID"]==inc_rd_list[i]]
        # print (df_gridsmart_routes_temp)

        # df_gridsmart_routes_temp2 = df_gridsmart_routes_temp.dropna(inplace=True)
        # df_gridsmart_routes_temp2 = df_gridsmart_routes_temp[df_gridsmart_routes_temp["OpenDriveToID"]!=null]
        df_gridsmart_routes_temp2= df_gridsmart_routes_temp[~df_gridsmart_routes_temp.isnull().any(axis=1)]
        # intersection = df_gridsmart_routes_temp["IntersectionName"].unique()
        # print (intersection)
        # print (df_gridsmart_routes_temp2)
        route_movement_list = df_gridsmart_routes_temp2["Turn"].unique().tolist()
        print (route_movement_list)
        
        
    # for i in range(1):
        vissim_road_id = get_corresponding_vissim_id(inc_rd_list[i])
        vissim_approach_links_dict[route_movement_list[0][:2]]= vissim_road_id

        turn_movement_link_ids = get_route_end_vissim_id(vissim_road_id)
        print (turn_movement_link_ids)
        dict_movements = dict(zip(turn_movement_link_ids, route_movement_list))
        dict_movement_list.append(dict_movements)
        
    print (vissim_approach_links_dict)

    print (dict_movement_list)
    
    return dict_movement_list, vissim_approach_links_dict

    




In [418]:
get_corresponding_turn_movement_link_numbers(4)

[324, 298, 309, 312]
['SBR', 'SBT', 'SBL', 'SBU']
[10107, 10108, 10109, 10110]
4
[90, 91, 92, 93]
[90, 91, 92, 93]
['WBR', 'WBT', 'WBL', 'WBU']
[10038, 10039, 10040, 10041]
4
[94, 95, 96, 97]
[94, 95, 96, 97]
['NBR', 'NBT', 'NBL', 'NBU']
[10061, 10062, 10063, 10064]
4
[98, 99, 100, 101]
[98, 99, 100, 101]
['EBR', 'EBT', 'EBL', 'EBU']
[10069, 10070, 10071, 10072]
4
[102, 103, 104, 105]
[102, 103, 104, 105]
{'SB': 45, 'WB': 19, 'NB': 30, 'EB': 33}
[{90: 'SBR', 91: 'SBT', 92: 'SBL', 93: 'SBU'}, {94: 'WBR', 95: 'WBT', 96: 'WBL', 97: 'WBU'}, {98: 'NBR', 99: 'NBT', 100: 'NBL', 101: 'NBU'}, {102: 'EBR', 103: 'EBT', 104: 'EBL', 105: 'EBU'}]


([{90: 'SBR', 91: 'SBT', 92: 'SBL', 93: 'SBU'},
  {94: 'WBR', 95: 'WBT', 96: 'WBL', 97: 'WBU'},
  {98: 'NBR', 99: 'NBT', 100: 'NBL', 101: 'NBU'},
  {102: 'EBR', 103: 'EBT', 104: 'EBL', 105: 'EBU'}],
 {'SB': 45, 'WB': 19, 'NB': 30, 'EB': 33})

In [419]:
## for an intersection id/opendrive junction id get phase number, associated turn movement, 
def getPhasingandTimingInfoForIntersection(intid_no):
    skiprow_var = 582
    nrow_var = 1002
    intid = intid_no
    turn_phase_dict = {}
    dfSynchroSignalPhasingsRaw = pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Synchro_signal.csv", skiprows=skiprow_var-1, nrows=nrow_var-skiprow_var)
    # print (dfSynchroSignalPhasingsRaw)
    df_signal_intersection_temp = dfSynchroSignalPhasingsRaw[dfSynchroSignalPhasingsRaw["INTID"]==intid]
    turnMovementKeys = ["NBL", "NBT", "NBR", "SBL", "SBT", "SBR", "EBL", "EBT", "EBR", "WBL", "WBT", "WBR"]
    for i in turnMovementKeys: 
        phase_no=df_signal_intersection_temp.loc[df_signal_intersection_temp['RECORDNAME'] == "Phase1", i].iloc[0]
        det_phase = df_signal_intersection_temp.loc[df_signal_intersection_temp['RECORDNAME'] == "DetectPhase1", i].iloc[0]
        perm_phase_no = df_signal_intersection_temp.loc[df_signal_intersection_temp['RECORDNAME'] == "PermPhase1", i].iloc[0]
        det_size = df_signal_intersection_temp.loc[df_signal_intersection_temp['RECORDNAME'] == "DetectSize1", i].iloc[0]

        # print (phase_no)
        # print (det_phase)

        if pd.isna(phase_no):
            phase_no = "nan"
        else:
            phase_no = int(phase_no)


        if pd.isna(det_phase):
            det_phase="nan"
        else:
            det_phase =int(det_phase)
            det_size = int(det_size)

        if pd.isna(perm_phase_no):
            perm_phase_no = "nan"    
        else:
            perm_phase_no=int(perm_phase_no)
            det_phase =int(det_phase)
            det_size = int(det_size)

        turn_phase_dict[i] = [phase_no, det_phase, det_size, perm_phase_no]
            
    print (turn_phase_dict)

    ## get timing for the associated phase number - max green, yellow, and red
    skiprow_timings = 1090
    nrow_timings = 1282
    # intid = 4
    df_signal_timings_raw = pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Synchro_signal.csv", skiprows=skiprow_timings-1, nrows=nrow_timings-skiprow_timings)
    # print (df_signal_timings_raw)
    df_signal_timings_temp = df_signal_timings_raw[df_signal_timings_raw["INTID"]==intid]
    # print (df_signal_timings_temp)
    phase_timing_dict ={}
    for k, v in turn_phase_dict.items():
        print(k, v)
        if v[0] != "nan":
            columnname = "D"+str(v[0])
            # print (columnname)
            minGreen = df_signal_timings_temp.loc[df_signal_timings_temp['RECORDNAME'] == "MinGreen", columnname].iloc[0]
            maxGreen = df_signal_timings_temp.loc[df_signal_timings_temp['RECORDNAME'] == "MaxGreen", columnname].iloc[0]
            vehExtension = df_signal_timings_temp.loc[df_signal_timings_temp['RECORDNAME'] == "VehExt", columnname].iloc[0]
            yellow= df_signal_timings_temp.loc[df_signal_timings_temp['RECORDNAME']== "Yellow", columnname].iloc[0]
            allred= df_signal_timings_temp.loc[df_signal_timings_temp['RECORDNAME'] == "AllRed", columnname].iloc[0]
            recall_info = df_signal_timings_temp.loc[df_signal_timings_temp['RECORDNAME'] == "Recall", columnname].iloc[0]
            # det_phase = df_signal_timings_temp.loc[df_signal_timings_temp['RECORDNAME'] == "DetectPhase1", columnname].iloc[0]
            # det_size = df_signal_timings_temp.loc[df_signal_timings_temp['RECORDNAME'] == "DetectSize1", columnname].iloc[0]
            phase_timings = [minGreen, maxGreen, vehExtension, yellow, allred, recall_info]
            v.append(phase_timings)
        else:
            v.append(["notimings"])
    print (turn_phase_dict)
    return turn_phase_dict
            
# The dictionary provides 
# turn movement: [phase number, detector number, permitted phase number,[phase number time - MaxGreen, phase number time - Yellow, phase number time - AllRed]]

In [420]:
getPhasingandTimingInfoForIntersection(4)

{'NBL': ['nan', 8, 20, 8], 'NBT': [8, 8, 50, 'nan'], 'NBR': [1, 1, 50, 8], 'SBL': ['nan', 4, 50, 4], 'SBT': [4, 4, 50, 'nan'], 'SBR': ['nan', 'nan', nan, 'nan'], 'EBL': [5, 5, 50, 2], 'EBT': [2, 2, 50, 'nan'], 'EBR': ['nan', 'nan', nan, 'nan'], 'WBL': [1, 1, 50, 6], 'WBT': [6, 6, 50, 'nan'], 'WBR': ['nan', 'nan', nan, 'nan']}
NBL ['nan', 8, 20, 8]
NBT [8, 8, 50, 'nan']
NBR [1, 1, 50, 8]
SBL ['nan', 4, 50, 4]
SBT [4, 4, 50, 'nan']
SBR ['nan', 'nan', nan, 'nan']
EBL [5, 5, 50, 2]
EBT [2, 2, 50, 'nan']
EBR ['nan', 'nan', nan, 'nan']
WBL [1, 1, 50, 6]
WBT [6, 6, 50, 'nan']
WBR ['nan', 'nan', nan, 'nan']
{'NBL': ['nan', 8, 20, 8, ['notimings']], 'NBT': [8, 8, 50, 'nan', [8.0, 29.0, 2.0, 4.0, 1.0, 0.0]], 'NBR': [1, 1, 50, 8, [4.0, 7.0, 1.0, 4.0, 0.0, 0.0]], 'SBL': ['nan', 4, 50, 4, ['notimings']], 'SBT': [4, 4, 50, 'nan', [8.0, 29.0, 2.0, 4.0, 1.0, 0.0]], 'SBR': ['nan', 'nan', nan, 'nan', ['notimings']], 'EBL': [5, 5, 50, 2, [4.0, 7.0, 1.0, 4.0, 0.0, 0.0]], 'EBT': [2, 2, 50, 'nan', [15.0, 50

{'NBL': ['nan', 8, 20, 8, ['notimings']],
 'NBT': [8, 8, 50, 'nan', [8.0, 29.0, 2.0, 4.0, 1.0, 0.0]],
 'NBR': [1, 1, 50, 8, [4.0, 7.0, 1.0, 4.0, 0.0, 0.0]],
 'SBL': ['nan', 4, 50, 4, ['notimings']],
 'SBT': [4, 4, 50, 'nan', [8.0, 29.0, 2.0, 4.0, 1.0, 0.0]],
 'SBR': ['nan', 'nan', nan, 'nan', ['notimings']],
 'EBL': [5, 5, 50, 2, [4.0, 7.0, 1.0, 4.0, 0.0, 0.0]],
 'EBT': [2, 2, 50, 'nan', [15.0, 50.0, 2.0, 4.0, 1.0, 1.0]],
 'EBR': ['nan', 'nan', nan, 'nan', ['notimings']],
 'WBL': [1, 1, 50, 6, [4.0, 7.0, 1.0, 4.0, 0.0, 0.0]],
 'WBT': [6, 6, 50, 'nan', [15.0, 50.0, 2.0, 4.0, 1.0, 1.0]],
 'WBR': ['nan', 'nan', nan, 'nan', ['notimings']]}

In [421]:
## Function to create rbc template
import json 

def create_new_rbc_file(intid, newfilename):
    turn_movement_dict = getPhasingandTimingInfoForIntersection(intid)
    print (turn_movement_dict)

    with open(r'C:\Users\ets\Desktop\Projects\RT_ScenarioGenerator\Simulation\vissim_dev_tests\shallowford\template_v3.prbc', 'r') as f:
        dictionary_rbc_template = json.load(f)

    list_of_phases = []
    list_of_turn_keys =[]
    for item in turn_movement_dict: 
        phase_number = turn_movement_dict[item][0]
        print (phase_number)
        if phase_number=="nan":
            pass
        else:
            list_of_phases.append(phase_number)
            list_of_turn_keys.append(item)

    print (list_of_turn_keys)
    print (list_of_phases)
    print (list_of_phases.index(1))


    for k in range(8):
        if k+1 in list_of_phases:
            index_turn_key = list_of_phases.index(k+1)
            min_green = turn_movement_dict[list_of_turn_keys[index_turn_key]][4][0]
            max_green = turn_movement_dict[list_of_turn_keys[index_turn_key]][4][1]
            veh_extension = turn_movement_dict[list_of_turn_keys[index_turn_key]][4][2]
            
            yellow = turn_movement_dict[list_of_turn_keys[index_turn_key]][4][3]
            allred = turn_movement_dict[list_of_turn_keys[index_turn_key]][4][4]
            recall= turn_movement_dict[list_of_turn_keys[index_turn_key]][4][5]
            # if recall == 1:
            #     recall_info = "true"
            # else:
            #     recall_info = "false"
            det_phase = turn_movement_dict[list_of_turn_keys[index_turn_key]][1]
            det_size = turn_movement_dict[list_of_turn_keys[index_turn_key]][2]

            
            dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["MaxGreen1"] = int(round(max_green)*10)
            dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["MinGreen"] = int(round(min_green)*10)
            difference_round = round(max_green)-max_green
            print (round(difference_round,1))

            dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["Yellow"] = int((yellow-round(difference_round,1))*10)
            dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["RedClearance"] = int(allred*10)
            dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["VehExtension"] = int(veh_extension*10)
            dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["MinRecall"] = bool(recall)
            detector_dictionary={"CalledSGs": [det_phase], "ExtendedSGs": [det_phase], "ID": det_phase}
            dictionary_rbc_template["Controller"]["VehicleDetectors"].append(detector_dictionary)
            # dictionary_rbc_template["Controller"]["VehicleDetectors"][k]["ExtendedSGs"][0] = det_phase
            # dictionary_rbc_template["Controller"]["VehicleDetectors"][k]["ID"] = det_phase

        else: 
            dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["MaxGreen1"] = 0
            dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["MinGreen"] = 0
            dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["Yellow"] = 0
            dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["RedClearance"] = 0
            # dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["VehExtension"] = 0
            # dictionary_rbc_template["Controller"]["VehicleSignalGroups"][k]["MinRecall"] = 
            # dictionary_rbc_template["Controller"]["VehicleDetectors"][k]["CalledSGs"] = det_phase
            # dictionary_rbc_template["Controller"]["VehicleDetectors"][k]["ExtendedSGs"] = det_phase
            # dictionary_rbc_template["Controller"]["VehicleDetectors"][k]["CalledSGs"] = det_phase

    dictionary_rbc_template["Controller"]["ExecutionFrequency"]= 10
    json_text = json.dumps(dictionary_rbc_template)
    # new_dictionary_filename = "rbc_timings_4.prbc"
    with open("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\"+newfilename, "w") as f:
        f.write(json_text)
    return (list_of_phases)

        

In [422]:

list_of_phases = create_new_rbc_file(11, "rbc_timings_11_test.prbc")
unique_phase_list = np.unique(list_of_phases)
print(unique_phase_list)


{'NBL': ['nan', 'nan', nan, 'nan'], 'NBT': ['nan', 'nan', nan, 'nan'], 'NBR': ['nan', 'nan', nan, 'nan'], 'SBL': ['nan', 'nan', nan, 'nan'], 'SBT': ['nan', 'nan', nan, 'nan'], 'SBR': ['nan', 'nan', nan, 'nan'], 'EBL': [1, 1, 50, 6], 'EBT': [6, 6, 50, 'nan'], 'EBR': ['nan', 'nan', nan, 'nan'], 'WBL': ['nan', 'nan', nan, 'nan'], 'WBT': [2, 2, 50, 'nan'], 'WBR': ['nan', 2, 0, 2]}
NBL ['nan', 'nan', nan, 'nan']
NBT ['nan', 'nan', nan, 'nan']
NBR ['nan', 'nan', nan, 'nan']
SBL ['nan', 'nan', nan, 'nan']
SBT ['nan', 'nan', nan, 'nan']
SBR ['nan', 'nan', nan, 'nan']
EBL [1, 1, 50, 6]
EBT [6, 6, 50, 'nan']
EBR ['nan', 'nan', nan, 'nan']
WBL ['nan', 'nan', nan, 'nan']
WBT [2, 2, 50, 'nan']
WBR ['nan', 2, 0, 2]
{'NBL': ['nan', 'nan', nan, 'nan', ['notimings']], 'NBT': ['nan', 'nan', nan, 'nan', ['notimings']], 'NBR': ['nan', 'nan', nan, 'nan', ['notimings']], 'SBL': ['nan', 'nan', nan, 'nan', ['notimings']], 'SBT': ['nan', 'nan', nan, 'nan', ['notimings']], 'SBR': ['nan', 'nan', nan, 'nan', ['no

In [423]:
def create_signal_controller(intid, supplyfile):
# intid = 4
# create signal controller
    SignalController = Vissim.Net.SignalControllers.AddSignalController(intid) # unsigned int Key
    SignalController.SetAttValue("Type", 20)
    SignalController.SetAttValue("SupplyFile1 ", supplyfile)

    for k in range(8):
        SignalController.SGs.AddSignalGroup(k+1) # unsigned int Key
        SignalController.SGs.ItemByKey(k+1).SetAttValue("Name", "Signal group "+str(k+1))
    




In [424]:
def get_last_signal_head_number():
    Attributes1 = ("Name", "No")
    signal_head_numbers = list(Vissim.Net.SignalHeads.GetMultipleAttributes(Attributes1))
    
    if len(signal_head_numbers)==0:
        last_signal_head = 0
    else:
        last_signal_head = signal_head_numbers[len(signal_head_numbers)-1][1]
    print (last_signal_head)
    return last_signal_head

In [425]:
def get_last_stop_sign_number():
    Attributes1 = ("Name", "No")
    stop_sign_numbers = list(Vissim.Net.StopSigns.GetMultipleAttributes(Attributes1))
    
    if len(stop_sign_numbers)==0:
        last_stop_sign = 0
    else:
        last_stop_sign = stop_sign_numbers[len(stop_sign_numbers)-1][1]
    print (last_stop_sign)
    return last_stop_sign

In [426]:
def get_last_detector_number():
    Attributes1 = ("Name", "No")
    detector_numbers = list(Vissim.Net.Detectors.GetMultipleAttributes(Attributes1))
    
    if len(detector_numbers)==0:
        last_det_number = 0
    else:
        last_det_number = detector_numbers[len(detector_numbers)-1][1]
    print (last_det_number)
    return last_det_number

In [184]:
get_last_detector_number()

0


0

In [427]:
## add signal head
# def add_signal_head(sh_no, link_no, sg, orsg):
def add_signal_head(link_no, sg, orsg):
    Link = Vissim.Net.Links.ItemByKey(link_no)
    Lane_count = Link.AttValue("NumLanes")
    sh_no = get_last_signal_head_number()+1
    for i in range(Lane_count):
        Lane = Link.Lanes.ItemByKey(i+1)
        Vissim.Net.SignalHeads.AddSignalHead(sh_no+i, Lane, 1)
        Vissim.Net.SignalHeads.ItemByKey(sh_no+i).SetAttValue("SG", sg)
        # print ("HERE")
        print (sh_no, sg)
        if orsg != "None":
            Vissim.Net.SignalHeads.ItemByKey(sh_no+i).SetAttValue("OrSG", orsg)
            print (sh_no, orsg)
        print(Lane_count)
    


In [428]:
def add_detectors_for_actuated_signal(lane, pos, type, length, sc, sg): #rightmost_lane, det_pos, 1, det_length, intid, det_port_no
    # detector = Vissim.Net.Detectors.AddObject(link_number, link_coord)
    det_no = get_last_detector_number()+1
    # Link = Vissim.Net.Links.ItemByKey(link_no)
    # lane = Link.Lanes.ItemByKey(lane_no)
    name = str(sc)+"_"+str(sg)+"_"+str(det_no)
    # det_length = 
    # port_no = 
    detector = Vissim.Net.Detectors.AddDetector(det_no, lane, 15)
    detector.SetAttValue('Type', type)
    detector.SetAttValue('Length', length)
    detector.SetAttValue('Name', name)
    detector.SetAttValue("SC", sc)
    detector.SetAttValue("PortNo", sg)
    detector.SetAttValue("Pos", pos)

    

In [429]:
def adjustDetector(type, length, pos, portno, sg, sc):
    det_no = get_last_detector_number()
    name = str(sc)+"_"+str(sg)+"_"+str(det_no)
    detector = Vissim.Net.Detectors.ItemBykey(det_no)
    detector.SetAttValue('Type', type)
    detector.SetAttValue('Length', length)
    detector.SetAttValue('Name', name)
    detector.SetAttValue("SC", sc)
    detector.SetAttValue("PortNo", sg)
    detector.SetAttValue("Pos", pos)

In [430]:
def add_signal_head_and_detector(link_no, sg, orsg, type, det_length, det_pos, port_no, sc):
    Link = Vissim.Net.Links.ItemByKey(link_no)
    Lane_count = Link.AttValue("NumLanes")
    sh_no = get_last_signal_head_number()+1
    for i in range(Lane_count):
        Lane = Link.Lanes.ItemByKey(i+1)
        Vissim.Net.SignalHeads.AddSignalHead(sh_no+i, Lane, 1)
        Vissim.Net.SignalHeads.ItemByKey(sh_no+i).SetAttValue("SG", sg)
        add_detectors_for_actuated_signal(Lane, det_pos, type, det_length, sc, port_no)
        # print ("HERE")
        print (sh_no, sg)
        if orsg != "None":
            Vissim.Net.SignalHeads.ItemByKey(sh_no+i).SetAttValue("OrSG", orsg)
            print (sh_no, orsg)
        print(Lane_count)
        return Lane_count

In [54]:
add_signal_head(22,90,"5-1","5-2")

In [431]:
Link = Vissim.Net.Links.ItemByKey(21)
linkLength = Link.AttValue("Length2D")
Lane = Link.Lanes.ItemByKey(1)
print (Lane)
det_length = 10
# print ("det_length", det_length)
det_pos = linkLength -det_length #adjsut det position if needed
print ("DET POS", det_pos)
detector = Vissim.Net.Detectors.AddDetector(1, Lane, 10) 
detector.SetAttValue("Length", 10)
detector.SetAttValue("Pos", 210)

<win32com.gen_py.Vissim Object Library 22.0 64 Bit.ILane instance at 0x1557133822128>
DET POS 216.0932480217468


com_error: (-2147352567, 'Exception occurred.', (0, 'VISSIM.Vissim.2200', 'CCOMDetectorContainerBase<CDetectorManager>::AddDetector: Detector: Network object key 1 already exists in network.', None, 0, -2147352567), None)

In [432]:
def add_stop_sign(link_no, sg):
    Link = Vissim.Net.Links.ItemByKey(link_no)
    Lane_count = Link.AttValue("NumLanes")
    Lane = Link.Lanes.ItemByKey(1)
    stop_no = get_last_stop_sign_number()+1
    Vissim.Net.StopSigns.AddStopSign(stop_no, Lane, 2)
    Vissim.Net.StopSigns.ItemByKey(stop_no).SetAttValue("SG", sg)
    return

In [433]:
def create_sc_sg_and_place_sh(intid):
    # creats signal heads and deploy to signal controllers 
    turn_movement_dict = get_corresponding_turn_movement_link_numbers(intid)[0]
    approach_ids = get_corresponding_turn_movement_link_numbers(intid)[1]
    # print (turn_movement_dict)
    phasing_timing_dict = getPhasingandTimingInfoForIntersection(intid)
    # print (phasing_timing_dict)
    create_signal_controller(intid, "rbc_timings_"+str(intid)+".prbc")

    for i in range(len(turn_movement_dict)):
        # print (i)
        # print (turn_movement_dict)
        
        # sh_no = (i+1)*25
        small_dict = turn_movement_dict[i]
        # print (small_dict)
        dict_key_list = list(small_dict.keys())
        # print (dict_key_list)

        approach_id_raw= small_dict[dict_key_list[0]]
        # print (approach_id_raw)
        approach_id_now = approach_id_raw[:2]
        # print(approach_id_now)
        approach_link_no = approach_ids[approach_id_now]
        print ("approach link no", approach_link_no)
        Link = Vissim.Net.Links.ItemByKey(approach_link_no)
    
        linkLength = Link.AttValue("Length2D")
        Lane_count = Link.AttValue("NumLanes")
        print (Lane_count)
        
        lanes = []
        for i in range(Lane_count):
            print ("lanes", Link.Lanes.ItemByKey(1))
            Lane = Link.Lanes.ItemByKey(i+1)
            print (Lane)
            lanes.append(Lane)
        # print ("lanes HERE",len(lanes))
        # lanes.pop(0)
        # print ("lnes after", len(lanes))

        for k in range(len(dict_key_list)):
            link_no = dict_key_list[k]
            turn = small_dict[link_no]
            print (link_no)
            print (turn)
            # phase_no = phasing_timing_dict[turn][0]
            # sg = str(intid)+"-"+str(phase_no)
            

            if turn[-1] != "U":
                phase_no = phasing_timing_dict[turn][0]
                det_port_no = phasing_timing_dict[turn][1]
                det_length = phasing_timing_dict[turn][2]

                if det_length == "nan":
                    det_length = 15

                if linkLength-det_length<5:
                    det_length = 4
                
                # det_length = 20
                print ("det_length", det_length)
                det_pos = linkLength-det_length #adjsut det position if needed
                print ("DET POS", det_pos)
                perm_phase_no = phasing_timing_dict[turn][3]
                print (turn, phase_no, perm_phase_no)

                if turn[-1] =="R":
                    turn_approach = turn[0:2]
                    
                    if phasing_timing_dict[turn_approach+"T"][0]!= "nan":
                        phase_no_stop = phasing_timing_dict[turn_approach+"T"][0]
                        print (phase_no_stop)
                        sg_stop = str(intid)+"-"+str(phase_no_stop)
                        print ("RIGHT")
                        print ("HEREEEEE111")
                        # for right_link_no, age in small_dict.items():
                        #     if age == turn_approach+"R":
                        #         print ("HERE")
                        #         print (link_no)
                        #     else:
                        #         link_no="nan"
                    # link_no
                        # print (link_no)
                        # if link_no != "nan":
                        print (link_no, sg_stop)
                        add_stop_sign(link_no, sg_stop)
                        print (link_no, " STOP ADDED")
                        add_signal_head(link_no, sg_stop, "None")
                        if len(lanes)>0:
                            print ("HEREEEEE111", det_length)
                            rightmost_lane = lanes[0]
                            if det_port_no!="nan":
                                add_detectors_for_actuated_signal(rightmost_lane, det_pos, 1, det_length, intid, det_port_no) #rightmost lane
                                lanes.pop(0)

                
                else:
                    if phase_no != "nan" and perm_phase_no !="nan":
                        print ("HERE222")
                        sg = str(intid)+"-"+str(phase_no)
                        orsg = str(intid)+"-"+str(perm_phase_no)
                        # add_signal_head(sh_no+5*(k+1), link_no, sg, orsg)
                        add_signal_head(link_no, sg, orsg)
                        if len(lanes)>0:
                            print ("HERE222")
                            rightmost_lane = lanes[0]
                            print (rightmost_lane)
                            
                            add_detectors_for_actuated_signal(rightmost_lane, det_pos, 1, det_length, intid, det_port_no) #rightmostlane
                            lanes.pop(0)


                    elif phase_no=="nan" and perm_phase_no!="nan":
                        sg = str(intid)+"-"+str(perm_phase_no)
                        print ("HERE3333")
                        # add_signal_head(sh_no+5*(k+1), link_no, sg, "None")
                        add_signal_head(link_no, sg, "None")
                        if len(lanes)>0:
                            print ("HERE3333")
                            rightmost_lane = lanes[0]
                            
                            add_detectors_for_actuated_signal(rightmost_lane, det_pos, 1, det_length, intid, det_port_no) #nextlaneavailable 
                            lanes.pop(0)
                        

                    elif phase_no!="nan" and perm_phase_no=="nan":
                        sg = str(intid)+"-"+str(phase_no)
                        print ("HEREEEEEE",)
                        # add_signal_head(sh_no+5*(k+1), link_no, sg, "None")
                        add_signal_head(link_no, sg, "None")
                        if len(lanes)>0:
                            # print ("HEREEEEEE")
                            rightmost_lane = lanes[0]
                            
                            add_detectors_for_actuated_signal(rightmost_lane, det_pos, 1, det_length, intid, det_port_no) #nextlaneavailable
                            lanes.pop(0)
                            print (sg)
                    
                    
                    else:
                        pass
            
                



                

            

        

# create signal head

# assign signal head - sc-sg values

# go to other dict -> get value for the movement, if it has own phase/permissive phase put that; if it has own and permissive put both; 
# for i in turn_movement_dict:
#     print (i,turn_movement_dict[i])


    
    #get phasing and timing information for the turn movement


# getPhasingandTimingInfoForIntersection(4)


In [365]:
create_sc_sg_and_place_sh(4)

[324, 298, 309, 312]
['SBR', 'SBT', 'SBL', 'SBU']
[10107, 10108, 10109, 10110]
4
[90, 91, 92, 93]
[90, 91, 92, 93]
['WBR', 'WBT', 'WBL', 'WBU']
[10038, 10039, 10040, 10041]
4
[94, 95, 96, 97]
[94, 95, 96, 97]
['NBR', 'NBT', 'NBL', 'NBU']
[10061, 10062, 10063, 10064]
4
[98, 99, 100, 101]
[98, 99, 100, 101]
['EBR', 'EBT', 'EBL', 'EBU']
[10069, 10070, 10071, 10072]
4
[102, 103, 104, 105]
[102, 103, 104, 105]
{'SB': 45, 'WB': 19, 'NB': 30, 'EB': 33}
[{90: 'SBR', 91: 'SBT', 92: 'SBL', 93: 'SBU'}, {94: 'WBR', 95: 'WBT', 96: 'WBL', 97: 'WBU'}, {98: 'NBR', 99: 'NBT', 100: 'NBL', 101: 'NBU'}, {102: 'EBR', 103: 'EBT', 104: 'EBL', 105: 'EBU'}]
[324, 298, 309, 312]
['SBR', 'SBT', 'SBL', 'SBU']
[10107, 10108, 10109, 10110]
4
[90, 91, 92, 93]
[90, 91, 92, 93]
['WBR', 'WBT', 'WBL', 'WBU']
[10038, 10039, 10040, 10041]
4
[94, 95, 96, 97]
[94, 95, 96, 97]
['NBR', 'NBT', 'NBL', 'NBU']
[10061, 10062, 10063, 10064]
4
[98, 99, 100, 101]
[98, 99, 100, 101]
['EBR', 'EBT', 'EBL', 'EBU']
[10069, 10070, 10071, 1

In [376]:
## Deploy signal heads on all intersections
list_of_int_ids = [4, 8, 11, 22, 19, 16]
for i in range(len(list_of_int_ids)):
    print ("this intersection", i)
    create_new_rbc_file(list_of_int_ids[i], "rbc_timings_"+str(list_of_int_ids[i])+".prbc")
    create_sc_sg_and_place_sh(list_of_int_ids[i])


this intersection 0
{'NBL': ['nan', 8, 20, 8], 'NBT': [8, 8, 50, 'nan'], 'NBR': [1, 1, 50, 8], 'SBL': ['nan', 4, 50, 4], 'SBT': [4, 4, 50, 'nan'], 'SBR': ['nan', 'nan', nan, 'nan'], 'EBL': [5, 5, 50, 2], 'EBT': [2, 2, 50, 'nan'], 'EBR': ['nan', 'nan', nan, 'nan'], 'WBL': [1, 1, 50, 6], 'WBT': [6, 6, 50, 'nan'], 'WBR': ['nan', 'nan', nan, 'nan']}
NBL ['nan', 8, 20, 8]
NBT [8, 8, 50, 'nan']
NBR [1, 1, 50, 8]
SBL ['nan', 4, 50, 4]
SBT [4, 4, 50, 'nan']
SBR ['nan', 'nan', nan, 'nan']
EBL [5, 5, 50, 2]
EBT [2, 2, 50, 'nan']
EBR ['nan', 'nan', nan, 'nan']
WBL [1, 1, 50, 6]
WBT [6, 6, 50, 'nan']
WBR ['nan', 'nan', nan, 'nan']
{'NBL': ['nan', 8, 20, 8, ['notimings']], 'NBT': [8, 8, 50, 'nan', [8.0, 29.0, 2.0, 4.0, 1.0, 0.0]], 'NBR': [1, 1, 50, 8, [4.0, 7.0, 1.0, 4.0, 0.0, 0.0]], 'SBL': ['nan', 4, 50, 4, ['notimings']], 'SBT': [4, 4, 50, 'nan', [8.0, 29.0, 2.0, 4.0, 1.0, 0.0]], 'SBR': ['nan', 'nan', nan, 'nan', ['notimings']], 'EBL': [5, 5, 50, 2, [4.0, 7.0, 1.0, 4.0, 0.0, 0.0]], 'EBT': [2, 2, 

In [435]:
## Add demand/volume to the model
## Get list of entry links - function from previous work 
#import vissim link info file 
csv_file_name = 'C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\Simulation\\vissim_dev_tests\\shallowford\\combined_inventory.csv'
df_vissim_ink_info_raw = pd.read_csv(csv_file_name)
links_only_df = df_vissim_ink_info_raw[df_vissim_ink_info_raw['Link-or-Connector'] == 'possibly_link'] 
connectors_only_df = df_vissim_ink_info_raw[df_vissim_ink_info_raw['Link-or-Connector'] == 'possibly_connector'] 
# print (links_only_df)
link_list = links_only_df.Link_No.unique().tolist()
print (link_list)
connector_list = connectors_only_df.Link_No.unique().tolist()
# connectors_only_df[['ToLinkNo', 'ToLaneNo']] = connectors_only_df.ToLinkLane.str.split(" ", expand = True)
# print (connectors_only_df)

veh_input_link_list =[]
for i in range(len(link_list)):
    link_id = link_list[i]
    filtered_connector = connectors_only_df[connectors_only_df['ToLink']==str(link_id)]
    print (filtered_connector)
    if len(filtered_connector.index)== 0:
        print (link_id)
        veh_input_link_list.append(link_id)

print (veh_input_link_list)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 150, 151, 152, 153, 154, 155, 156, 157, 158, 164, 165, 166, 167, 168, 169, 170, 171, 172, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185]
Empty DataFrame
Columns: [Link_No, Link_Name, Ped_Area, Link-or-Connector, FromLinkLane, FromLinkPos, ToLinkLane, ToLinkPos, StartLinkCoord-X, StartLinkCoord-Y, StartLinkCoord-Z, Latitude, Longitude, FromLink, FromLane, ToLink, ToLane, Diver

In [436]:
remove_list = [52, 12, 41, 51, 37]
veh_input_link_list_final = [i for i in veh_input_link_list if i not in remove_list]
print (veh_input_link_list_final)

[1, 13, 29, 30, 32, 33, 38, 39, 40, 45, 46, 47]


In [434]:
def get_corresponding_opendrive_edgeID(vissim_link_id):
    # vissim_link_id = 1
    df = pd.read_csv(r'C:\Users\ets\Desktop\Projects\RT_ScenarioGenerator\Simulation\vissim_dev_tests\shallowford\vissim_link_opendrive_road_id_mapping.csv')
    df_filtered = df[df["Link_No"]==vissim_link_id]
    correspondingOpendriveID = df_filtered.iloc[0]["opendrive_road_id"]
    # print (correspondingOpendriveID)
    return correspondingOpendriveID


In [437]:
# function to get volume of time interval for opendrive ID
import math
def get_volume_vissim_link(opendriveFromID, interval_no):
# opendriveFromID = 309
    df_gridsmart_routes = pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_lookuptable_2.csv")
    df_gridsmart_demand =  pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_demand.csv")

    df_gridsmart_routes_temp = df_gridsmart_routes[df_gridsmart_routes["OpenDriveFromID"]==opendriveFromID]
    # print (df_gridsmart_routes_temp)
    intersection = df_gridsmart_routes_temp["IntersectionName"].unique()
    # print (intersection)
    # print (df_gridsmart_routes_temp)
    route_movement_list = df_gridsmart_routes_temp["Turn"].unique().tolist()
    # print (route_movement_list)
    df_demand_temp = df_gridsmart_demand[df_gridsmart_demand["IntersectionName"]==intersection[0]].reset_index()
    # print (df_demand_temp)
    total_demand = 0

    # demand = df_demand_temp.loc[interval_no-1, route_movement_list[0]]
    # print(df_demand_temp.loc[[interval_no-1]])
    # print (demand)
    # for j in range(start_interval, end_interval+1):
    for j in range(len(route_movement_list)):
        # print (start_interval)
        # print (route_movement_list[j])
        demand = df_demand_temp.loc[interval_no-1, route_movement_list[j]]
        # print (demand)
        x = float(demand)
        math.isnan(x)
        if math.isnan(x)==False:
            # print (route_movement_list[j])
            total_demand = total_demand+demand
    # print (total_demand)
    return total_demand

In [438]:
opendriveid = get_corresponding_opendrive_edgeID(1)
print (opendriveid)
get_volume_vissim_link(float(opendriveid), start_interval)



280


186.0

In [68]:
simtime = 3600
interval_duration = 15*60
no_intervals = int(simtime/interval_duration)-1
startTimeofDayinSec = 28800
start_interval = int(startTimeofDayinSec/interval_duration)+1
end_interval = start_interval+no_intervals
# print (end_interval)
for k in range(start_interval, end_interval+1):
    get_volume_vissim_link(280, k)

                    IntersectionName Turn  OpenDriveFromID  OpenDriveToID
92  Gunbarrel Road & Shallowford Rd.  WBR            280.0          306.0
93  Gunbarrel Road & Shallowford Rd.  WBT            280.0          307.0
94  Gunbarrel Road & Shallowford Rd.  WBL            280.0          306.0
95  Gunbarrel Road & Shallowford Rd.  WBU            280.0          304.0
['Gunbarrel Road & Shallowford Rd.']
    index                  IntersectionName   Time  NBR   NBT   NBL  NBU  \
0     384  Gunbarrel Road & Shallowford Rd.   0:00  NaN   2.0  12.0  NaN   
1     385  Gunbarrel Road & Shallowford Rd.   0:15  NaN   9.0  14.0  NaN   
2     386  Gunbarrel Road & Shallowford Rd.   0:30  NaN   5.0   5.0  NaN   
3     387  Gunbarrel Road & Shallowford Rd.   0:45  NaN   3.0  10.0  NaN   
4     388  Gunbarrel Road & Shallowford Rd.   1:00  2.0   2.0   9.0  NaN   
..    ...                               ...    ...  ...   ...   ...  ...   
91    475  Gunbarrel Road & Shallowford Rd.  22:45  4.0  18.0

In [439]:
## Function to create vehicle inputs 
def createVehicleInputs(veh_input_link_list_final):
    vissim_veh_input_keys_list = []
    for i in range(len(veh_input_link_list_final)):
        # if veh_input_link_list_final[i] == 32:
        #     veh_input_vissim_link = 31
        # else:
        veh_input_vissim_link = veh_input_link_list_final[i]
        vissim_link_object = Vissim.Net.Links.ItemByKey(veh_input_vissim_link)
        VehInput = Vissim.Net.VehicleInputs.AddVehicleInput(0, vissim_link_object)
        print (VehInput.AttValue('No'))
        vissim_veh_input_keys_list.append(VehInput.AttValue('No'))
    return vissim_veh_input_keys_list

In [442]:
p = createVehicleInputs(veh_input_link_list_final)

1
2
3
4
5
6
7
8
9
10
11
12


In [450]:
## Function to assign


simtime = 3600
interval_duration = 15*60
no_intervals = int(simtime/interval_duration)-1
startTimeofDayinSec = 28800
start_interval = int(startTimeofDayinSec/interval_duration)+1
end_interval = start_interval+no_intervals
vissim_veh_input_keys_list = p
# print (start_interval)
# print (end_interval)

# for each entry link
for i in range(len(veh_input_link_list_final)):
    veh_input_link = veh_input_link_list_final[i]
    veh_inp_no = vissim_veh_input_keys_list[i]
    if veh_input_link == 32:
        veh_input_link = 31
    opendrive_link_id = get_corresponding_opendrive_edgeID(veh_input_link)
    print (veh_input_link)
    print (opendrive_link_id)
    # print (get_volume_vissim_link(opendrive_link_id, start_interval))
    
    for j in range(start_interval, end_interval+1):
        vol = get_volume_vissim_link(float(opendrive_link_id), j)
        t = j-start_interval+1
        print (t)

        
        #assign volume to the vehicle input time interval
        Vissim.Net.VehicleInputs.ItemByKey(veh_inp_no).SetAttValue('Volume('+str(t)+')', vol*4)
        #assign volume type to the vehicle input time interval
        Vissim.Net.VehicleInputs.ItemByKey(veh_inp_no).SetAttValue('VolType('+str(t)+')', 1)
    #         Vissim.Net.VehicleInputs.ItemByKey(VI_number).SetAttValue('Cont('+str(t)+')', False)

    

    
# get vehicle input number
# if entry link is 32, 31 
# find corresponding opendrive id
# for the range of interval 
# get the demand
# assign the demand to vehicle input 

1
280
1
2
3
4
13
292
1
2
3
4
29
308
1
2
3
4
30
309
1
2
3
4
31
310
1
2
3
4
33
312
1
2
3
4
38
317
1
2
3
4
39
318
1
2
3
4
40
319
1
2
3
4
45
324
1
2
3
4
46
325
1
2
3
4
47
326
1
2
3
4


In [61]:
## Function to get demand total for the link for the time interval
## This is a test cell
simtime = 3600
interval_duration = 15*60
no_intervals = int(simtime/interval_duration)-1
startTimeofDayinSec = 54000
start_interval = int(startTimeofDayinSec/interval_duration)+1
end_interval = start_interval+no_intervals

print (start_interval)
print (end_interval)

df_gridsmart_routes = pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_lookuptable_2.csv")
df_gridsmart_demand =  pd.read_csv("C:\\Users\\ets\\Desktop\\Projects\\RT_ScenarioGenerator\\GridSmart_demand.csv")

df_gridsmart_routes_temp = df_gridsmart_routes[df_gridsmart_routes["OpenDriveFromID"]==309]
intersection = df_gridsmart_routes_temp["IntersectionName"].unique()
print (intersection)
print (df_gridsmart_routes_temp)
route_movement_list = df_gridsmart_routes_temp["Turn"].unique().tolist()
print (route_movement_list)
df_demand_temp = df_gridsmart_demand[df_gridsmart_demand["IntersectionName"]==intersection[0]]
for j in range(start_interval, end_interval+1):
    demand = df_demand_temp.loc[j-1, route_movement_list[0]]
    print (demand)


61
64
['Amin Dr./Shallowford Village Dr. & Shallowford Rd.']
                                    IntersectionName Turn  OpenDriveFromID  \
0  Amin Dr./Shallowford Village Dr. & Shallowford...  NBR            309.0   
1  Amin Dr./Shallowford Village Dr. & Shallowford...  NBT            309.0   
2  Amin Dr./Shallowford Village Dr. & Shallowford...  NBL            309.0   
3  Amin Dr./Shallowford Village Dr. & Shallowford...  NBU            309.0   

   OpenDriveToID  
0          293.0  
1          296.0  
2          295.0  
3          294.0  
['NBR', 'NBT', 'NBL', 'NBU']
3.0
3.0
4.0
7.0


In [ ]:
## Function to create vehicle input, time interval, and assign the demand to it - by noon


## Test complete working - by evening Wednesday

## Netconvert - to opendrive file - Thursday

## Carmaker integration - Thursday

## Conflict marker - set up 